# RAG-ассистент поддержки онлайн-кинотеатра «Фильмоскоп»

Notebook для воспроизводимого прототипа LangChain + Chroma RAG-системы.

Основная логика должна импортироваться из `src/`.

## Этап 1. Аудит и подготовка корпуса

### 1.2. Первичный аудит корпуса

Цель ячеек — проверить структуру корпуса до очистки: категории, форматы, длины `raw`, признаки HTML/Markdown, mojibake, мусорные последовательности и дубли.

In [1]:
from src.config import load_settings
from src.data_loading import load_dataset
from src.diagnostics import audit_corpus, duplicate_groups_to_dataframe, print_audit_summary

settings = load_settings()
dataset = load_dataset(settings.data.dataset_path)
audit = audit_corpus(dataset.docs)

print_audit_summary(audit)

C:\PythonProg\ai-agent.rag-system\.venv\Lib\site-packages\urllib3\contrib\socks.py:50: DependencyWarning: SOCKS support in urllib3 requires the installation of optional dependencies: specifically, PySocks.  For more information, see https://urllib3.readthedocs.io/en/latest/advanced-usage.html#socks-proxies
  warnings.warn(


Документов: 181
Категорий: 5
Форматов: 3
Коротких документов: 11
Длинных документов: 10
Документов с HTML: 21
Документов с Markdown: 29
Документов с mojibake: 17
Документов с подозрительными фрагментами: 1
Групп дублей: 6


#### Распределения и длины

Таблицы ниже показывают баланс категорий, форматов и статистику длины исходного поля `raw`. Эти значения нужны для выбора стратегии очистки и будущего чанкинга.

In [2]:
display(audit.category_distribution)
display(audit.format_distribution)
display(audit.length_statistics)

category,count,share
Устройства и качество,42,0.232
Подписка и оплата,38,0.2099
Аккаунт и профили,35,0.1934
Функции и контент,34,0.1878
Ошибки воспроизведения,32,0.1768


format,count,share
text,132,0.7293
markdown,28,0.1547
html,21,0.116


metric,value
count,181
min,14
p05,237
p25,429
median,664
mean,804.14
p75,1048
p95,1764
max,3057


#### Короткие и длинные документы

Короткие документы нельзя удалять автоматически: часть из них может быть полезными объявлениями или заглушками. Длинные документы остаются в корпусе, но их нужно учесть при чанкинге.

In [10]:
display("Короткие документы:")
display(audit.short_documents[["doc_id", "title", "category", "format", "raw_length", "preview"]])
display("Длинные документы:")
display(audit.long_documents[["doc_id", "title", "category", "format", "raw_length", "preview"]])

'Короткие документы:'

doc_id,title,category,format,raw_length,preview
153,Таблица совместимости (фрагмент),Устройства и качество,text,14,QX-500;2020;да
156,Схема разделов (изображение),Функции и контент,html,42,"<div class=""kb-article""><h1>Схема разделов"
145,Обновление приложения при ошибках,Ошибки воспроизведения,text,70,"Если ошибка мешает просмотру, обновите приложение до последней версии."
53,Раздел перенесён,Функции и контент,text,110,"Раздел перенесён. См. приложение 2 к регламенту размещения справочных материалов. [черновик, не публиковать]"
71,Статья обновляется,Ошибки воспроизведения,html,129,"<div class=""kb-article""><h1>Статья обновляется</h1> <p>Раздел временно недоступен Статья обновляется. Загляните позже.</p> </div>"
74,Раздел в разработке,Устройства и качество,html,149,"<div class=""kb-article""><h1>Раздел в разработке</h1> <p>Раздел в разработке Мы готовим этот раздел. Информация появится в ближайшее время.</p> </div>"
52,Раздел справки (архив),Устройства и качество,html,167,"<div class=""kb-article""><h1>Раздел справки (архив)</h1> <p>Страница не найдена Возможно, раздел был перемещён или удалён. Воспользуйтесь поиском по справке.</p> </div>"
39,Экспорт обращений за неделю,Подписка и оплата,text,230,ticket_id;category;status;resolved_hours 10234;оплата;closed;2 10235;oплата;closed;5 10241;подпискa;open; 10250;возврат;closed;26 10252;оплaта;closed;1 10260;пpoмокод;closed;3 Экс…
75,Конфигурация player-api,Ошибки воспроизведения,text,231,"{""service"": ""player-api"", ""version"": ""2.14.3"", ""env"": ""production"", ""features"": {""subtitles_v2"": true, ""preload"": false}, ""updated_at"": ""2026-02-11T09:14:00Z""} Конфигурация выгруж…"
18,Плановые работы (объявление),Аккаунт и профили,text,237,ВНИМАНИЕ! В НОЧЬ НА ВТОРНИК ПРОВОДЯТСЯ ПЛАНОВЫЕ РАБОТЫ. ЛИЧНЫЙ КАБИНЕТ И НАСТРОЙКИ АККАУНТА БУДУТ НЕДОСТУПНЫ С 02:00 ДО 05:00 ПО МОСКОВСКОМУ ВРЕМЕНИ. ПРОСМОТР ФИЛЬМОВ ПРОДОЛЖИТ РА…


'Длинные документы:'

doc_id,title,category,format,raw_length,preview
73,Аренда и покупка фильмов,Функции и контент,markdown,3057,# ÐÑÐµÐ½Ð´Ð° Ð¸ Ð¿Ð¾ÐºÑÐ¿ÐºÐ° ÑÐ¸Ð»ÑÐ¼Ð¾Ð² ÐÐ»Ð°ÑÐ½ÑÐµ ÑÐ¸Ð»ÑÐ¼Ñ Ð²Ð¸Ð´Ð½Ñ Ð¿ÑÑÐ¼Ð¾ Ð² ÐºÐ°ÑÐ°Ð»Ð¾Ð³Ðµ: Ð² ÐºÐ°ÑÑÐ¾ÑÐºÐµ Ð²Ð¼ÐµÑÑÐ¾ ÐºÐ½Ð¾Ð¿ÐºÐ¸ Â«Ð¡Ð¼Ð¾ÑÑÐµ…
159,Вход на телевизоре по коду,Устройства и качество,text,3017,"ÐÑ Ð¾Ð´ Ð¿Ð¾ ÐºÐ¾Ð´Ñ ÑÐ´ÐµÐ»Ð°Ð½ Ð´Ð»Ñ ÑÑÑÑÐ¾Ð¹ÑÑÐ², Ð³Ð´Ðµ Ð½ÐµÑ ÑÐ´Ð¾Ð±Ð½Ð¾Ð¹ ÐºÐ»Ð°Ð²Ð¸Ð°ÑÑÑÑ: ÑÐµÐ»ÐµÐ²Ð¸Ð·Ð¾ÑÐ¾Ð² Ð¸ Ð¿ÑÐ¸ÑÑÐ°Ð²Ð¾Ðº. ÐÐ° ÑÐµÐ»ÐµÑÐ¾Ð½…"
22,Объёмный звук 5.1,Устройства и качество,text,2961,ÐÐ±Ð¾Ð·Ð½Ð°ÑÐµÐ½Ð¸Ðµ 5.1 Ð¾Ð¿Ð¸ÑÑÐ²Ð°ÐµÑ ÑÐ°ÑÐºÐ»Ð°Ð´ÐºÑ ÐºÐ°Ð½Ð°Ð»Ð¾Ð²: Ð¿ÑÑÑ Ð¾ÑÐ½Ð¾Ð²Ð½ÑÑ ÐºÐ¾Ð»Ð¾Ð½Ð¾Ðº Ð²Ð¾ÐºÑÑÐ³ Ð·ÑÐ¸ÑÐµÐ»Ñ Ð¸ Ð¾ÑÐ´ÐµÐ»ÑÐ½ÑÐ¹ ÐºÐ°Ð½Ð°…
173,Что делать при подозрении на взлом,Аккаунт и профили,html,2818,"<!DOCTYPE html><html><head><title>Что делать при подозрении на взлом — Справка «Фильмоскопа»</title><meta name=""viewport"" content=""width=device-width""></head><body> <nav class=""ma…"
110,Что нужно для просмотра в 4K HDR,Устройства и качество,html,2798,"<!DOCTYPE html><html><head><title>Что нужно для просмотра в 4K HDR — Справка «Фильмоскопа»</title><meta name=""viewport"" content=""width=device-width""></head><body> <nav class=""main…"
155,Пробный период,Подписка и оплата,html,2566,"<!DOCTYPE html><html><head><title>Пробный период — Справка «Фильмоскопа»</title><meta name=""viewport"" content=""width=device-width""></head><body> <nav class=""main-menu""><a href=""/""…"
99,Совместный просмотр,Функции и контент,html,2564,"<!DOCTYPE html><html><head><title>Совместный просмотр — Справка «Фильмоскопа»</title><meta name=""viewport"" content=""width=device-width""></head><body> <nav class=""main-menu""><a hre…"
113,Тарифы «Фильмоскопа»,Подписка и оплата,markdown,1862,"# Тарифы «Фильмоскопа» Тариф выбирается при оформлении подписки и в любой момент виден в настройках аккаунта, в разделе «Подписка»: там указаны название текущего тарифа, дата след…"
144,Скачивание фильмов и сериалов,Функции и контент,markdown,1800,"# Скачивание фильмов и сериалов Загрузки нужны там, где на связь рассчитывать не приходится: в самолёте, в поезде, в поездке с дорогим мобильным трафиком. Скачанный фильм открывае…"
47,Ошибка P-103: контент недоступен в регионе,Ошибки воспроизведения,html,1764,"<!DOCTYPE html><html><head><title>Ошибка P-103: контент недоступен в регионе — Справка «Фильмоскопа»</title><meta name=""viewport"" content=""width=device-width""></head><body> <nav c…"


#### Разметка, кодировка и подозрительные последовательности

Эти таблицы фиксируют проблемы, которые будут обработаны на этапе 1.3. HTML и Markdown нужно очищать без потери смысла, mojibake исправлять только при уверенной эвристике, а подозрительные последовательности проверять вручную.

In [4]:
display("Документы с html разметкой:")
display(audit.html_documents[["doc_id", "title", "category", "format", "raw_length", "preview"]].head(20))
display("Документы с markdown разметкой:")
display(audit.markdown_documents[["doc_id", "title", "category", "format", "raw_length", "preview"]].head(20))
display("Документы с mojibake разметкой:")
display(audit.mojibake_documents[["doc_id", "title", "category", "format", "mojibake_markers", "preview"]])
display("Документы с suspicious разметкой:")
display(audit.suspicious_documents[["doc_id", "title", "category", "format", "suspicious_sequences", "preview"]])

'Документы с html разметкой:'

doc_id,title,category,format,raw_length,preview
8,Пользовательское соглашение (выдержка): контент,Функции и контент,html,471,"<div class=""kb-article""><h1>Пользовательское соглашение (выдержка): контент</h1> <p>Пользовательское соглашение (выдержка). Раздел 9: контент 9.1. Все материалы сервиса охраняются…"
19,«Фильмоскоп Бокс»: скоро,Устройства и качество,html,372,"<div class=""kb-article""><h1>«Фильмоскоп Бокс»: скоро</h1> <p>«Фильмоскоп Бокс»: скоро Мы готовим собственную ТВ-приставку — «Фильмоскоп Бокс». Пульт с горячей кнопкой сервиса, быс…"
23,Ошибка P-107 (браузер),Ошибки воспроизведения,html,778,"<div class=""kb-article""><h1>Ошибка P-107 (браузер)</h1> <p>Ошибка P-107 в браузере Ошибка P-107 появляется, когда браузер не может воспроизвести защищенное видео. Сообщение показы…"
32,Почему зрители выбирают «Максимум»,Подписка и оплата,html,399,"<div class=""kb-article""><h1>Почему зрители выбирают «Максимум»</h1> <p>Почему зрители выбирают «Максимум» «Максимум» — это кино без компромиссов: лучшее, что может дать домашний э…"
47,Ошибка P-103: контент недоступен в регионе,Ошибки воспроизведения,html,1764,"<!DOCTYPE html><html><head><title>Ошибка P-103: контент недоступен в регионе — Справка «Фильмоскопа»</title><meta name=""viewport"" content=""width=device-width""></head><body> <nav c…"
52,Раздел справки (архив),Устройства и качество,html,167,"<div class=""kb-article""><h1>Раздел справки (архив)</h1> <p>Страница не найдена Возможно, раздел был перемещён или удалён. Воспользуйтесь поиском по справке.</p> </div>"
57,Политика обработки персональных данных (выдержка),Аккаунт и профили,html,504,"<div class=""kb-article""><h1>Политика обработки персональных данных (выдержка)</h1> <p>Политика обработки персональных данных (выдержка) Сервис обрабатывает данные пользователей дл…"
71,Статья обновляется,Ошибки воспроизведения,html,129,"<div class=""kb-article""><h1>Статья обновляется</h1> <p>Раздел временно недоступен Статья обновляется. Загляните позже.</p> </div>"
74,Раздел в разработке,Устройства и качество,html,149,"<div class=""kb-article""><h1>Раздел в разработке</h1> <p>Раздел в разработке Мы готовим этот раздел. Информация появится в ближайшее время.</p> </div>"
78,Пользовательское соглашение (выдержка): подписка и платежи,Подписка и оплата,html,585,"<div class=""kb-article""><h1>Пользовательское соглашение (выдержка): подписка и платежи</h1> <p>Пользовательское соглашение (выдержка). Раздел 4: подписка и платежи 4.1. Оформляя п…"


'Документы с markdown разметкой:'

doc_id,title,category,format,raw_length,preview
2,Безопасность аккаунта: общие советы,Аккаунт и профили,markdown,460,"# Безопасность аккаунта: общие советы Несколько привычек, которые защищают аккаунт лучше любых настроек: используйте уникальный пароль, который не повторяется на других сайтах; не…"
6,10 фильмов для уютного вечера,Функции и контент,markdown,432,"# 10 фильмов для уютного вечера 10 фильмов для уютного вечера: выбор редакции Собрали проверенные картины, которые согревают лучше пледа: тёплые комедии, добрые драмы и пара мюзик…"
24,Какой тариф подойдёт именно вам,Подписка и оплата,markdown,1155,"# Какой тариф подойдёт именно вам В «Фильмоскопе» три тарифа: «Мобильный», «Стандарт» и «Максимум». Они отличаются числом одновременных просмотров, максимальным качеством картинки…"
25,Тарифы «Фильмоскопа» (архив),Подписка и оплата,markdown,486,"# Тарифы «Фильмоскопа» (архив) Внимание: статья устарела и хранится в архиве. Актуальные цены и условия — в действующей статье «Тарифы ""Фильмоскопа""». Линейка тарифов до марта 202…"
34,«Фильмоскоп Дети»,Функции и контент,markdown,440,"# «Фильмоскоп Дети» «Фильмоскоп Дети» — отдельный раздел каталога с мультфильмами, детским кино и обучающими программами. Весь контент раздела имеет рейтинг 0+ или 6+ и проходит р…"
45,Требования для просмотра в 4K (архив),Устройства и качество,markdown,420,# Требования для просмотра в 4K (архив) Внимание: статья устарела и хранится в архиве. Актуальные требования — в действующей статье о просмотре в 4K HDR. На момент запуска функции…
48,Мультфильмы для самых маленьких,Функции и контент,markdown,962,"# Мультфильмы для самых маленьких Редакция собрала подборку для дошкольников: мультфильмы, которые можно включить трёхлетнему зрителю и не сидеть рядом с пультом наготове. Отбирал…"
56,4K и HDR: что это такое,Устройства и качество,markdown,998,"# 4K и HDR: что это такое Две характеристики картинки, которые упоминают вместе, хотя отвечают они за разное: 4K — про количество точек, HDR — про яркость и цвет. 4K вчетверо подр…"
58,Протокол встречи контент-команды,Подписка и оплата,text,459,"Протокол рабочей встречи контент-команды, 14 марта. Присутствовали: редакция базы знаний, продукт, поддержка. 1. Провести ревизию статей раздела «Подписка и оплата» — до конца ква…"
59,Трансляция и дублирование экрана: в чём разница,Устройства и качество,markdown,1297,"# Трансляция и дублирование экрана: в чём разница Оба способа выводят содержимое телефона на большой экран, но устроены они по-разному, и путать их не стоит. Дублирование экрана (…"


'Документы с mojibake разметкой:'

doc_id,title,category,format,mojibake_markers,preview
22,Объёмный звук 5.1,Устройства и качество,text,"Â, Ð, Ñ",ÐÐ±Ð¾Ð·Ð½Ð°ÑÐµÐ½Ð¸Ðµ 5.1 Ð¾Ð¿Ð¸ÑÑÐ²Ð°ÐµÑ ÑÐ°ÑÐºÐ»Ð°Ð´ÐºÑ ÐºÐ°Ð½Ð°Ð»Ð¾Ð²: Ð¿ÑÑÑ Ð¾ÑÐ½Ð¾Ð²Ð½ÑÑ ÐºÐ¾Ð»Ð¾Ð½Ð¾Ðº Ð²Ð¾ÐºÑÑÐ³ Ð·ÑÐ¸ÑÐµÐ»Ñ Ð¸ Ð¾ÑÐ´ÐµÐ»ÑÐ½ÑÐ¹ ÐºÐ°Ð½Ð°…
38,Объединение аккаунтов и перенос подписки,Аккаунт и профили,text,"Â, Ð, Ñ","ÐÐ±ÑÐµÐ´Ð¸Ð½Ð¸ÑÑ Ð´Ð²Ð° Ð°ÐºÐºÐ°ÑÐ½ÑÐ° Â«Ð¤Ð¸Ð»ÑÐ¼Ð¾ÑÐºÐ¾Ð¿Ð°Â» Ð² Ð¾Ð´Ð¸Ð½ Ð½ÐµÐ²Ð¾Ð·Ð¼Ð¾Ð¶Ð½Ð¾: Ð¸ÑÑÐ¾ÑÐ¸Ñ Ð¿ÑÐ¾ÑÐ¼Ð¾ÑÑÐ¾Ð², ÑÐ¿Ð¸ÑÐºÐ¸ Â«ÐÐ¾ÑÂ», Ð¿ÑÐ¾ÑÐ¸Ð…"
43,Аудиодорожки и оригинальная озвучка,Функции и контент,text,"Â, Ð, Ñ","Ð£ Ð¼Ð½Ð¾Ð³Ð¸Ñ ÑÐ¸Ð»ÑÐ¼Ð¾Ð² Ð¸ ÑÐµÑÐ¸Ð°Ð»Ð¾Ð² ÐµÑÑÑ Ð½ÐµÑÐºÐ¾Ð»ÑÐºÐ¾ Ð°ÑÐ´Ð¸Ð¾Ð´Ð¾ÑÐ¾Ð¶ÐµÐº: ÑÑÑÑÐºÐ°Ñ Ð¾Ð·Ð²ÑÑÐºÐ°, Ð¾ÑÐ¸Ð³Ð¸Ð½Ð°Ð»ÑÐ½Ð°Ñ Ð´Ð¾ÑÐ¾Ð¶ÐºÐ°, Ð¸Ð…"
46,Памятка по оценкам,Функции и контент,text,"Â, Ð, Ñ","ÐÑÐ°ÑÐºÐ°Ñ Ð¿Ð°Ð¼ÑÑÐºÐ° Ð¿Ð¾ Ð¾ÑÐµÐ½ÐºÐ°Ð¼: ÑÑÐ°Ð²ÑÑÐµ Â«ÐÑÐ°Ð²Ð¸ÑÑÑÂ» ÑÐ¸Ð»ÑÐ¼Ð°Ð¼, ÐºÐ¾ÑÐ¾ÑÑÐµ Ñ Ð¾ÑÐµÑÑÑ ÑÐ¾Ð²ÐµÑÐ¾Ð²Ð°ÑÑ Ð´ÑÑÐ·ÑÑÐ¼, Ð¸ Â«ÐÐµ…"
54,Приложение зависает на заставке,Ошибки воспроизведения,text,"Ð, Ñ","ÐÑÐ»Ð¸ Ð¿ÑÐ¸Ð»Ð¾Ð¶ÐµÐ½Ð¸Ðµ Ð·Ð°Ð²Ð¸ÑÐ°ÐµÑ Ð½Ð° Ð·Ð°ÑÑÐ°Ð²ÐºÐµ Ð¸ Ð½Ðµ Ð¾ÑÐºÑÑÐ²Ð°ÐµÑ Ð³Ð»Ð°Ð²Ð½ÑÐ¹ ÑÐºÑÐ°Ð½, Ð·Ð°ÐºÑÐ¾Ð¹ÑÐµ ÐµÐ³Ð¾ Ð¿Ð¾Ð»Ð½Ð¾ÑÑÑÑ Ð¸ Ð·Ð°Ð¿ÑÑ…"
55,Восстановление пароля,Аккаунт и профили,text,"Â, Ð, Ñ","ÐÑÐ»Ð¸ Ð½Ðµ Ð¿Ð¾Ð»ÑÑÐ°ÐµÑÑÑ Ð²Ð¾Ð¹ÑÐ¸, Ð½Ð°Ð¶Ð¼Ð¸ÑÐµ Â«ÐÐ°Ð±ÑÐ»Ð¸ Ð¿Ð°ÑÐ¾Ð»Ñ?Â» Ð½Ð° ÑÐºÑÐ°Ð½Ðµ Ð²Ñ Ð¾Ð´Ð°. ÐÐ¾ÑÑÑÐ°Ð½Ð¾Ð²Ð»ÐµÐ½Ð¸Ðµ ÑÐ°Ð±Ð¾ÑÐ°ÐµÑ ÑÐµÑÐµÐ·…"
60,Wi-Fi или кабель: как подключить телевизор,Устройства и качество,text,"Ð, Ñ","ï»¿Ð¡ÑÐ°Ð±Ð¸Ð»ÑÐ½Ð¾ÑÑÑ Ð¿ÑÐ¾ÑÐ¼Ð¾ÑÑÐ° ÑÐ¸Ð»ÑÐ½Ð¾ Ð·Ð°Ð²Ð¸ÑÐ¸Ñ Ð¾Ñ ÑÐ¾Ð³Ð¾, ÐºÐ°Ðº ÑÐµÐ»ÐµÐ²Ð¸ï»¿Ð·Ð¾Ñ Ð¿Ð¾Ð´ÐºÐ»ÑÑÑÐ½ Ðº ÑÐµÑÐ¸.â Ð¡Ð°Ð¼ÑÐ¹ Ð½Ð°Ð´ÑÐ¶Ð½Ñ…"
73,Аренда и покупка фильмов,Функции и контент,markdown,"Â, Ð, Ñ",# ÐÑÐµÐ½Ð´Ð° Ð¸ Ð¿Ð¾ÐºÑÐ¿ÐºÐ° ÑÐ¸Ð»ÑÐ¼Ð¾Ð² ÐÐ»Ð°ÑÐ½ÑÐµ ÑÐ¸Ð»ÑÐ¼Ñ Ð²Ð¸Ð´Ð½Ñ Ð¿ÑÑÐ¼Ð¾ Ð² ÐºÐ°ÑÐ°Ð»Ð¾Ð³Ðµ: Ð² ÐºÐ°ÑÑÐ¾ÑÐºÐµ Ð²Ð¼ÐµÑÑÐ¾ ÐºÐ½Ð¾Ð¿ÐºÐ¸ Â«Ð¡Ð¼Ð¾ÑÑÐµ…
93,Голосовое управление на телевизоре,Устройства и качество,text,"Â, Ð, Ñ","Ð£Ð¿ÑÐ°Ð²Ð»ÑÑÑ Ð¿ÑÐ¸Ð»Ð¾Ð¶ÐµÐ½Ð¸ÐµÐ¼ Ð½Ð° ÑÐµÐ»ÐµÐ²Ð¸Ð·Ð¾ÑÐµ Ð¼Ð¾Ð¶Ð½Ð¾ Ð³Ð¾Ð»Ð¾ÑÐ¾Ð¼, ÐµÑÐ»Ð¸ Ð¿ÑÐ»ÑÑ Ð¿Ð¾Ð´Ð´ÐµÑÐ¶Ð¸Ð²Ð°ÐµÑ Ð³Ð¾Ð»Ð¾ÑÐ¾Ð²Ð¾Ð¹ Ð²Ð²Ð¾Ð´: ÑÐ´ÐµÑÐ¶…"
106,Чеки и история платежей,Подписка и оплата,text,"Â, Ð, Ñ","ÐÑÑÐ¾ÑÐ¸Ñ Ð²ÑÐµÑ Ð¿Ð»Ð°ÑÐµÐ¶ÐµÐ¹ Ñ ÑÐ°Ð½Ð¸ÑÑÑ Ð² ÑÐ°Ð·Ð´ÐµÐ»Ðµ Â«ÐÐ»Ð°ÑÐµÐ¶Ð¸Â» Ð½Ð°ÑÑÑÐ¾ÐµÐº Ð°ÐºÐºÐ°ÑÐ½ÑÐ°: ÑÐ°Ð¼ Ð²Ð¸Ð´Ð½Ñ Ð´Ð°ÑÐ°, ÑÑÐ¼Ð¼Ð° Ð¸ ÑÐ¿Ð¾Ñ…"


'Документы с suspicious разметкой:'

doc_id,title,category,format,suspicious_sequences,preview
120,Уведомления,Аккаунт и профили,text,\ufffd,"«Фильмоскоп» присылает два типа уведомлений. Сервисные — о списаниях, входах в аккаунт и изменениях подписки — отключить н�льзя: они защища�т акк�унт и деньги. Рекламные — о новин…"


#### Полные и почти полные дубли

Группы дублей не удаляются автоматически. Если группы найдены, решение о фильтрации или объединении принимается после ручного просмотра.

In [5]:
duplicate_groups_df = duplicate_groups_to_dataframe(audit.duplicate_groups)
display(duplicate_groups_df)

doc_ids,titles,similarity,match_type
"77, 175",Как смотреть с телефона на большом экране | Трансляция через Chromecast и AirPlay,1.0,exact
"0, 95",Промокоды | Промокоды и скидки,0.9892,near
"7, 121",Удаление аккаунта | Как удалить аккаунт,0.9582,near
"23, 160",Ошибка P-107 (браузер) | Ошибка P-107 в браузере,0.9909,near
"83, 151",Как зарегистрироваться | Регистрация аккаунта,0.9972,near
"105, 158",Поиск фильмов и сериалов | Поиск по каталогу,0.9815,near


#### Решения по найденным проблемам

Сводка ниже фиксирует действие для каждого класса проблем: исправлять, фильтровать или оставить до ручной проверки.

In [6]:
display(audit.decisions)

problem,count,decision
HTML-разметка,21,исправлять на этапе 1.3: извлечь читаемый текст без потери ссылок и смысловых блоков
Markdown-разметка,29,"исправлять на этапе 1.3: убрать служебные символы Markdown, сохранив структуру списков и заголовков"
mojibake/ошибки кодировки,17,исправлять только при уверенной эвристике; спорные фрагменты помечать для ручной проверки
мусорные символы и подозрительные последовательности,1,исправлять безопасные случаи нормализацией пробелов и удалением управляющих символов; остальное оставить до ручной проверки
короткие документы,11,"не удалять автоматически; проверить, являются ли это заглушки, объявления или полезные короткие статьи"
длинные документы,10,оставить в корпусе и учитывать при выборе стратегии чанкинга
полные или почти полные дубли,6,фильтровать или объединять только после ручной проверки совпадающих групп


### 1.3. Очистка и нормализация корпуса

Цель ячеек — построить очищенный корпус без удаления документов и без потери полезной структуры:

- HTML извлекается библиотечным ingest-подходом: `trafilatura` как основной extractor, `BeautifulSoup` + `lxml` как fallback с удалением boilerplate DOM-узлов.
- Markdown не преобразуется в plain text на этапе очистки: заголовки, списки, ссылки, таблицы и inline-code сохраняются для следующего Markdown-aware chunking.
- Mojibake исправляется через `ftfy` только если диагностические признаки уверенно улучшаются.
- Регулярные выражения используются только для финальной нормализации пробелов, управляющих символов, переносов по дефису и диагностики остаточного мусора.


In [7]:
from src.cleaning import clean_corpus
from src.diagnostics import (
    build_cleaning_examples_table,
    build_cleaning_operations_table,
    compare_corpus_quality,
)

cleaned_results = clean_corpus(dataset.docs)
cleaned_docs = [result.cleaned_document for result in cleaned_results]
cleaned_audit = audit_corpus(cleaned_docs)

display(f"Очищено документов: {len(cleaned_docs)}")
display(build_cleaning_operations_table(cleaned_results))


'Очищено документов: 181'

operation,documents
unchanged,110
whitespace_normalized,35
markdown_marked_for_chunking,30
html_cleaned,21
html_trafilatura,20
mojibake_fixed,16
html_bs4_fallback,1


#### Сравнение качества «было → стало»

Таблица ниже показывает, как изменились ключевые признаки качества после очистки. Количество документов должно сохраниться: на этом этапе данные не удаляются.


In [8]:
quality_comparison = compare_corpus_quality(audit, cleaned_audit)
display(quality_comparison)

display("Оставшиеся документы с признаками HTML:")
display(cleaned_audit.html_documents[["doc_id", "title", "format", "raw_length", "preview"]])
display("Оставшиеся документы с признаками Markdown:")
display(cleaned_audit.markdown_documents[["doc_id", "title", "format", "raw_length", "preview"]])
display("Оставшиеся документы с mojibake-маркерами:")
display(cleaned_audit.mojibake_documents[["doc_id", "title", "format", "raw_length", "mojibake_markers", "preview"]])


metric,before,after,delta
documents,181,181,0
html_documents,21,0,-21
markdown_documents,29,30,1
mojibake_documents,17,1,-16
suspicious_documents,1,1,0
duplicate_groups,6,7,1
min_raw_length,14,14,0
median_raw_length,664,574,-90
max_raw_length,3057,1861,-1196


'Оставшиеся документы с признаками HTML:'

doc_id,title,format,raw_length,preview


'Оставшиеся документы с признаками Markdown:'

doc_id,title,format,raw_length,preview
2,Безопасность аккаунта: общие советы,markdown,459,"# Безопасность аккаунта: общие советы Несколько привычек, которые защищают аккаунт лучше любых настроек: используйте уникальный пароль, который не повторяется на других сайтах; не…"
6,10 фильмов для уютного вечера,markdown,431,"# 10 фильмов для уютного вечера 10 фильмов для уютного вечера: выбор редакции Собрали проверенные картины, которые согревают лучше пледа: тёплые комедии, добрые драмы и пара мюзик…"
24,Какой тариф подойдёт именно вам,markdown,1154,"# Какой тариф подойдёт именно вам В «Фильмоскопе» три тарифа: «Мобильный», «Стандарт» и «Максимум». Они отличаются числом одновременных просмотров, максимальным качеством картинки…"
25,Тарифы «Фильмоскопа» (архив),markdown,485,"# Тарифы «Фильмоскопа» (архив) Внимание: статья устарела и хранится в архиве. Актуальные цены и условия — в действующей статье «Тарифы ""Фильмоскопа""». Линейка тарифов до марта 202…"
34,«Фильмоскоп Дети»,markdown,439,"# «Фильмоскоп Дети» «Фильмоскоп Дети» — отдельный раздел каталога с мультфильмами, детским кино и обучающими программами. Весь контент раздела имеет рейтинг 0+ или 6+ и проходит р…"
45,Требования для просмотра в 4K (архив),markdown,419,# Требования для просмотра в 4K (архив) Внимание: статья устарела и хранится в архиве. Актуальные требования — в действующей статье о просмотре в 4K HDR. На момент запуска функции…
48,Мультфильмы для самых маленьких,markdown,961,"# Мультфильмы для самых маленьких Редакция собрала подборку для дошкольников: мультфильмы, которые можно включить трёхлетнему зрителю и не сидеть рядом с пультом наготове. Отбирал…"
56,4K и HDR: что это такое,markdown,997,"# 4K и HDR: что это такое Две характеристики картинки, которые упоминают вместе, хотя отвечают они за разное: 4K — про количество точек, HDR — про яркость и цвет. 4K вчетверо подр…"
58,Протокол встречи контент-команды,text,459,"Протокол рабочей встречи контент-команды, 14 марта. Присутствовали: редакция базы знаний, продукт, поддержка. 1. Провести ревизию статей раздела «Подписка и оплата» — до конца ква…"
59,Трансляция и дублирование экрана: в чём разница,markdown,1296,"# Трансляция и дублирование экрана: в чём разница Оба способа выводят содержимое телефона на большой экран, но устроены они по-разному, и путать их не стоит. Дублирование экрана (…"


'Оставшиеся документы с mojibake-маркерами:'

doc_id,title,format,raw_length,mojibake_markers,preview
120,Уведомления,text,446,�,"«Фильмоскоп» присылает два типа уведомлений. Сервисные — о списаниях, входах в аккаунт и изменениях подписки — отключить н�льзя: они защища�т акк�унт и деньги. Рекламные — о новин…"


#### Примеры документов до и после очистки

Примеры отсортированы по величине изменения длины текста, чтобы быстро увидеть наиболее заметные преобразования.


In [9]:
cleaning_examples = build_cleaning_examples_table(cleaned_results, max_examples=12)
display(cleaning_examples)


doc_id,title,format,operations,length_before,length_after,before_preview,after_preview
73,Аренда и покупка фильмов,markdown,"mojibake_fixed, markdown_marked_for_chunking, whitespace_normalized",3057,1679,# ÐÑÐµÐ½Ð´Ð° Ð¸ Ð¿Ð¾ÐºÑÐ¿ÐºÐ° ÑÐ¸Ð»ÑÐ¼Ð¾Ð² ÐÐ»Ð°ÑÐ½ÑÐµ ÑÐ¸Ð»ÑÐ¼Ñ Ð²Ð¸Ð´Ð½Ñ Ð¿ÑÑÐ¼Ð¾ Ð² ÐºÐ°ÑÐ°Ð»Ð¾Ð³Ðµ: Ð² ÐºÐ°ÑÑÐ¾ÑÐºÐµ Ð²Ð¼ÐµÑÑÐ¾ ÐºÐ½Ð¾Ð¿ÐºÐ¸ Â«Ð¡Ð¼Ð¾ÑÑÐµ…,"# Аренда и покупка фильмов Платные фильмы видны прямо в каталоге: в карточке вместо кнопки «Смотреть» появляются «Арендовать» и «Купить», а рядом указана цена. Оформлять что-то от…"
159,Вход на телевизоре по коду,text,mojibake_fixed,3017,1657,"ÐÑ Ð¾Ð´ Ð¿Ð¾ ÐºÐ¾Ð´Ñ ÑÐ´ÐµÐ»Ð°Ð½ Ð´Ð»Ñ ÑÑÑÑÐ¾Ð¹ÑÑÐ², Ð³Ð´Ðµ Ð½ÐµÑ ÑÐ´Ð¾Ð±Ð½Ð¾Ð¹ ÐºÐ»Ð°Ð²Ð¸Ð°ÑÑÑÑ: ÑÐµÐ»ÐµÐ²Ð¸Ð·Ð¾ÑÐ¾Ð² Ð¸ Ð¿ÑÐ¸ÑÑÐ°Ð²Ð¾Ðº. ÐÐ° ÑÐµÐ»ÐµÑÐ¾Ð½…","Вход по коду сделан для устройств, где нет удобной клавиатуры: телевизоров и приставок. На телефоне, планшете и в браузере вход выполняется обычным способом — почтой и паролем от…"
22,Объёмный звук 5.1,text,mojibake_fixed,2961,1622,ÐÐ±Ð¾Ð·Ð½Ð°ÑÐµÐ½Ð¸Ðµ 5.1 Ð¾Ð¿Ð¸ÑÑÐ²Ð°ÐµÑ ÑÐ°ÑÐºÐ»Ð°Ð´ÐºÑ ÐºÐ°Ð½Ð°Ð»Ð¾Ð²: Ð¿ÑÑÑ Ð¾ÑÐ½Ð¾Ð²Ð½ÑÑ ÐºÐ¾Ð»Ð¾Ð½Ð¾Ðº Ð²Ð¾ÐºÑÑÐ³ Ð·ÑÐ¸ÑÐµÐ»Ñ Ð¸ Ð¾ÑÐ´ÐµÐ»ÑÐ½ÑÐ¹ ÐºÐ°Ð½Ð°…,"Обозначение 5.1 описывает раскладку каналов: пять основных колонок вокруг зрителя и отдельный канал низких частот. За счёт этого шаги, дождь или проезжающая машина слышны с той ст…"
173,Что делать при подозрении на взлом,html,"html_cleaned, html_trafilatura, markdown_marked_for_chunking",2818,1817,"<!DOCTYPE html><html><head><title>Что делать при подозрении на взлом — Справка «Фильмоскопа»</title><meta name=""viewport"" content=""width=device-width""></head><body> <nav class=""ma…","Что делать при подозрении на взлом Что делать при подозрении на взлом Чужой доступ появляется не из-за взлома сервиса, а почти всегда проще: пароль повторяется на другом сайте, от…"
110,Что нужно для просмотра в 4K HDR,html,"html_cleaned, html_trafilatura",2798,1802,"<!DOCTYPE html><html><head><title>Что нужно для просмотра в 4K HDR — Справка «Фильмоскопа»</title><meta name=""viewport"" content=""width=device-width""></head><body> <nav class=""main…","Что нужно для просмотра в 4K HDR Что нужно для просмотра в 4K HDR 4K и HDR — разные вещи, хотя обычно идут вместе. 4K отвечает за детализацию: изображение состоит из заметно больш…"
47,Ошибка P-103: контент недоступен в регионе,html,"html_cleaned, html_trafilatura",1764,799,"<!DOCTYPE html><html><head><title>Ошибка P-103: контент недоступен в регионе — Справка «Фильмоскопа»</title><meta name=""viewport"" content=""width=device-width""></head><body> <nav c…","Ошибка P-103: контент недоступен в регионе Ошибка P-103 появляется, когда выбранный фильм или сериал недоступен в вашем текущем регионе. Сообщение показывается в плеере, просмотр…"
99,Совместный просмотр,html,"html_cleaned, html_trafilatura",2564,1599,"<!DOCTYPE html><html><head><title>Совместный просмотр — Справка «Фильмоскопа»</title><meta name=""viewport"" content=""width=device-width""></head><body> <nav class=""main-menu""><a hre…","Совместный просмотр Функция рассчитана на тех, кто находится далеко друг от друга. Если вы смотрите кино в одной комнате, отдельный сеанс не нужен: достаточно запустить фильм на о…"
155,Пробный период,html,"html_cleaned, html_trafilatura",2566,1606,"<!DOCTYPE html><html><head><title>Пробный период — Справка «Фильмоскопа»</title><meta name=""viewport"" content=""width=device-width""></head><body> <nav class=""main-menu""><a href=""/""…","Пробный период Начинается всё с обычного оформления: зарегистрируйте аккаунт, выберите тариф и укажите способ оплаты. Карта нужна сразу — она понадобится для будущих списаний, но…"
154,"Что делать, если оплата не прошла",html,"html_cleaned, html_trafilatura",1520,583,"<!DOCTYPE html><html><head><title>Что делать, если оплата не прошла — Справка «Фильмоскопа»</title><meta name=""viewport"" content=""width=device-width""></head><body> <nav class=""mai…","Что делать, если оплата не

#### Вывод по этапу 1.3

- Очищенный корпус построен без удаления документов: все `doc_id` сохранены для последующих retrieval-метрик.
- HTML очищается через `trafilatura`; для коротких или нестандартных страниц используется fallback `BeautifulSoup` + `lxml` с удалением служебных DOM-блоков и извлечением `article`/`main`.
- Markdown-разметка сохраняется как полезная структура документа и помечается операцией `markdown_marked_for_chunking` для Markdown-aware chunking на этапе 2.2.
- Целевой показатель `markdown_documents: 29 → 0` больше не используется: наличие Markdown после очистки ожидаемо.
- Mojibake исправляется через `ftfy` только при уверенном улучшении диагностик; документы с символом `�` не восстанавливаются автоматически, потому что исходные байты уже потеряны.
- Короткие, длинные и дублирующиеся документы не удаляются на этом этапе: решение о фильтрации/объединении откладывается до retrieval-экспериментов и ручной проверки влияния на качество.
